# Librerías

In [1]:
import yaml
from propiedad import Propiedad  # o desde tu clase base `Requester` si no necesitas parsear JSON todavía

# Cargar yaml

In [3]:
# Carga del user-agent personalizado si es externo
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:137.0) Gecko/20100101 Firefox/137.0"

# Leer el archivo YAML
with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Reemplazar user-agent si está marcado con placeholder
for paso in config.values():
    if paso["headers"].get("User-Agent") == "__USER_AGENT__":
        paso["headers"]["User-Agent"] = user_agent


# Funciones

In [ ]:

def obtener_urls_propiedades(subsection):
    return [
        "https://century21mexico.com" + r["urlCorrectaPropiedad"].replace('\\/', '/') + "?json=true"
        for r in subsection
        if "urlCorrectaPropiedad" in r and r["urlCorrectaPropiedad"]
    ]

# Acceder Home

In [18]:
# Crear instancia del requester (heredada)
prop = Propiedad()

# Paso 1: acceso a home
prop.get_requests(config["paso1"]["url"], config["paso1"]["headers"])

prop.last_response

HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF0CD0>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF0F50>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1450>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
No se obtuvo respuesta para: https://century21mexico.com/


# Acceder por filtros

### diccionarios

In [15]:
estados_mexico = [
    "/en-estado_aguascalientes",
    "/en-estado_baja-california",
    "/en-estado_baja-california-sur",
    "/en-estado_campeche",
    "/en-estado_chiapas",
    "/en-estado_chihuahua",
    "/en-estado_ciudad-de-mexico",
    "/en-estado_coahuila",
    "/en-estado_colima",
    "/en-estado_durango",
    "/en-estado_guanajuato",
    "/en-estado_guerrero",
    "/en-estado_hidalgo",
    "/en-estado_jalisco",
    "/en-estado_estado-de-mexico",
    "/en-estado_michoacan",
    "/en-estado_morelos",
    "/en-estado_nayarit",
    "/en-estado_nuevo-leon",
    "/en-estado_oaxaca",
    "/en-estado_puebla",
    "/en-estado_queretaro",
    "/en-estado_quintana-roo",
    "/en-estado_san-luis-potosi",
    "/en-estado_sinaloa",
    "/en-estado_sonora",
    "/en-estado_tabasco",
    "/en-estado_tamaulipas",
    "/en-estado_tlaxcala",
    "/en-estado_veracruz",
    "/en-estado_yucatan",
    "/en-estado_zacatecas"
]


precioss = [
"/precio-hasta_1000000",
"/precio-desde_1000001/precio-hasta_4000000",
"/precio-desde_4000001/precio-hasta_10000000",
"/precio-desde_10000001"
]

### acceder publicación json

In [16]:
# Paso 2: carga del JSON
prop.get_requests(config["paso2"]["url"], config["paso2"]["headers"])
prop.get_json()

print(prop._json)


HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1D10>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1BD0>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1590>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
{'pathContacto': 'https://century21mexico.com/interfaz/wh/c21wpLocal', 'totalHitsRelation': 'eq', 'totalHits': '22,684', 'total

## obtener urls por filtros

In [17]:
urls_por_estado = {}

for estado in range(0,32):

    total_urls =[]

    print(estados_mexico[estado])

    prop.get_requests("https://century21mexico.com/v/resultados" + "/en-pais_mexico" + estados_mexico[estado] + "?json=true", config["paso2"]["headers"]) 
    # Buscar el primer <script> que contenga la palabra totalHits
    try:
        prop.get_json()
        prop.get_subsection("totalHits", "0")
        total_hits = int(str(prop.subsection).replace(",", ""))
    except json.JSONDecodeError:
        print("No se encontró el JSON")
        total_hits = 0

    print(total_hits)

    if total_hits > 1500:
        print("vamoh a utilizar las divisiones")
        for precio in precioss:
            prop.get_requests("https://century21mexico.com/v/resultados" + "/en-pais_mexico" + estados_mexico[estado] + precio + "?json=true", config["paso2"]["headers"])
            try:
                prop.get_json()
                prop.get_subsection("totalHits", "0")
                total_hits = int(str(prop.subsection).replace(",", ""))
            except json.JSONDecodeError:
                print("No se encontró el JSON")
                total_hits = 0

            print(total_hits)
            paginas = math.ceil(total_hits / 100)
            print(paginas)

            prop.get_subsection("results", [])

            total_urls.extend(obtener_urls_propiedades(prop.subsection))

            if paginas > 1 :
        
                for page in range(2, paginas+1):#puede ser más de 15 xd
                    prop.get_requests("https://century21mexico.com/v/resultados" + "/en-pais_mexico" + estados_mexico[estado] + precio + "/pagina_" + str(page) + "?json=true", config["paso2"]["headers"])
                    prop.get_subsection("results", [])
                    total_urls.extend(obtener_urls_propiedades(prop.subsection))

    else:
        total_urls.extend(obtener_urls_propiedades(response))
        paginas = math.ceil(total_hits / 100)
        print(paginas)
        if paginas > 1 :
        
            for page in range(2, paginas+1):
                print(page)
                response = req.get_requests("https://century21mexico.com/v/resultados" + "/en-pais_mexico" + estados_mexico[estado] + "/pagina_" + str(page) + "?json=true", headers_json)
                prop.get_subsection("results", [])
                total_urls.extend(obtener_urls_propiedades(prop.subsection))

    

    ## Recorrer páginas /pagina_#
    

    #urls_por_estado[estado] = total_urls
    print(f"Se completó: {estados_mexico[estado]}")

    urls_por_estado[estados_mexico[estado]] = total_urls

    print(f"URL hasta ahora: {len(total_urls)}")
    

/en-estado_aguascalientes
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados/en-pais_mexico/en-estado_aguascalientes?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF16D0>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados/en-pais_mexico/en-estado_aguascalientes?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1E50>: Failed to resolve 'century21mexico.com' ([Errno 11001] getaddrinfo failed)"))
HTTPSConnectionPool(host='century21mexico.com', port=443): Max retries exceeded with url: /v/resultados/en-pais_mexico/en-estado_aguascalientes?json=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000272D3EF1950>: Failed to resolve 'century21mexico.com' ([Errno 11001] geta

NameError: name 'math' is not defined

# Obtener los atributos

In [ ]:
datos =[]
for estado in estados_mexico:
    print(f"Procesando estado: {estado}")
    datos.extend([
        obtener_datos(url)
        for url in urls_por_estado.get(estado, [])
    ])

df = pd.DataFrame(datos)

In [ ]:
df.head(100)

In [ ]:
df.to_csv('src/21century.csv', index=False)